# DSA Week 11 -- Shortest Paths (Dijkstra)

**Course:** Data Structures & Algorithms
**Session:** 3 hours
**Prerequisites:** Weeks 1-10 (especially heaps and graphs)
**Focus:** Dijkstra's algorithm, weighted graphs

## Learning Objectives

1. Understand weighted graphs and why BFS is not enough
2. Implement Dijkstra's algorithm using a priority queue
3. Trace Dijkstra step-by-step
4. Reconstruct the shortest path
5. Know Dijkstra's limitations (no negative weights)

## The Big Idea

BFS finds the shortest path by **edge count**. But in real life, edges have
different costs (distance, time, latency). Dijkstra's algorithm finds the
shortest path by **total weight**.

```
Weighted graph:

    A --4-- B
    |       |
    1       1
    |       |
    C --2-- D --5-- E

BFS shortest path A->D: A->B->D (2 edges)
Dijkstra shortest A->D: A->C->D (cost 1+2=3, vs A->B->D cost 4+1=5)

Dijkstra gives the CHEAPEST path, not the one with fewest edges.
```

In [ ]:
# === SETUP (run this first) ===
# If running in Google Colab, uncomment and run the lines below:
# !git clone https://github.com/ArifSolmaz/courseos-curriculum.git
# %cd courseos-curriculum/course-content

import sys, os
# Add src to path so we can import project modules
if os.path.exists('src'):
    sys.path.insert(0, 'src')
elif os.path.exists('../src'):
    sys.path.insert(0, '../src')
elif os.path.exists('../../src'):
    sys.path.insert(0, '../../src')

print("Setup complete! Ready to work.")

---
## Part 1: Dijkstra's Algorithm

Dijkstra works by always processing the **closest unvisited node** first
(using a priority queue / min-heap).

```
Algorithm:
  1. Set distance to start = 0, all others = infinity
  2. Add start to priority queue
  3. While queue not empty:
     a. Pop node with smallest distance
     b. For each neighbor:
        - Calculate new_dist = current_dist + edge_weight
        - If new_dist < known distance, update it
        - Add neighbor to queue with new_dist
  4. Return distances and paths
```

In [ ]:
import heapq

def dijkstra(graph, start, end=None):
    """Dijkstra's shortest path algorithm. O((V+E) log V).

    graph: {node: [(neighbor, weight), ...]}
    Returns: (distances, previous) for path reconstruction
    """
    distances = {start: 0}
    previous = {start: None}
    pq = [(0, start)]
    visited = set()

    while pq:
        dist, node = heapq.heappop(pq)

        if node in visited:
            continue
        visited.add(node)

        if node == end:
            break

        for neighbor, weight in graph.get(node, []):
            if neighbor not in visited:
                new_dist = dist + weight
                if new_dist < distances.get(neighbor, float("inf")):
                    distances[neighbor] = new_dist
                    previous[neighbor] = node
                    heapq.heappush(pq, (new_dist, neighbor))

    return distances, previous

def reconstruct_path(previous, start, end):
    """Reconstruct path from Dijkstra's previous dict."""
    path = []
    current = end
    while current is not None:
        path.append(current)
        current = previous.get(current)
    path.reverse()
    if path[0] != start:
        return []  # no path exists
    return path

---
## Part 2: Traced Example

In [ ]:
def dijkstra_traced(graph, start, end):
    """Dijkstra with step-by-step trace."""
    distances = {start: 0}
    previous = {start: None}
    pq = [(0, start)]
    visited = set()
    step = 0

    print("  Dijkstra from " + str(start) + " to " + str(end) + ":")
    print()

    while pq:
        dist, node = heapq.heappop(pq)

        if node in visited:
            continue

        step += 1
        visited.add(node)
        print("  Step " + str(step) + ": Process " + str(node) + " (distance=" + str(dist) + ")")

        if node == end:
            print("    Reached destination!")
            break

        for neighbor, weight in graph.get(node, []):
            if neighbor not in visited:
                new_dist = dist + weight
                old_dist = distances.get(neighbor, float("inf"))
                if new_dist < old_dist:
                    distances[neighbor] = new_dist
                    previous[neighbor] = node
                    heapq.heappush(pq, (new_dist, neighbor))
                    print("    -> " + str(neighbor) + ": " + str(dist) + " + " + str(weight) + " = " + str(new_dist) + (" (improved from " + str(old_dist) + ")" if old_dist < float("inf") else " (new)"))
                else:
                    print("    -> " + str(neighbor) + ": " + str(new_dist) + " >= " + str(old_dist) + " (no improvement)")

    path = reconstruct_path(previous, start, end)
    total = distances.get(end, float("inf"))
    print()
    print("  Shortest path: " + " -> ".join(str(n) for n in path))
    print("  Total distance: " + str(total))
    return total, path

# Example graph
graph = {
    "A": [("B", 4), ("C", 1)],
    "B": [("A", 4), ("D", 1), ("E", 7)],
    "C": [("A", 1), ("D", 2), ("F", 5)],
    "D": [("B", 1), ("C", 2), ("E", 3)],
    "E": [("B", 7), ("D", 3), ("F", 1)],
    "F": [("C", 5), ("E", 1)],
}

dijkstra_traced(graph, "A", "E")

---
## Part 3: Network Latency Example

In [ ]:
# Real-world example: finding fastest network route
network = {
    "Server": [("Router1", 2), ("Router2", 5)],
    "Router1": [("Server", 2), ("Router3", 3), ("Switch1", 1)],
    "Router2": [("Server", 5), ("Router3", 1), ("Switch2", 2)],
    "Router3": [("Router1", 3), ("Router2", 1), ("Switch1", 4), ("Switch2", 1)],
    "Switch1": [("Router1", 1), ("Router3", 4), ("SensorA", 1)],
    "Switch2": [("Router2", 2), ("Router3", 1), ("SensorB", 1)],
    "SensorA": [("Switch1", 1)],
    "SensorB": [("Switch2", 1)],
}

print("=== Finding fastest route: Server -> SensorB ===")
print()
dijkstra_traced(network, "Server", "SensorB")

print()
print("=== All distances from Server ===")
dists, _ = dijkstra(network, "Server")
for node in sorted(dists.keys()):
    print("  Server -> " + node + ": " + str(dists[node]) + " ms")

---
## Part 4: Dijkstra's Limitations

**Dijkstra does NOT work with negative edge weights.** If you need negative
weights, use the Bellman-Ford algorithm (not covered in this course).

```
Why negative weights break Dijkstra:

    A --(-3)-- B
    |          |
    1          2
    |          |
    C ---1---- D

Dijkstra from A: visits C first (dist=1), marks it done.
But A->B->D->C has total cost -3+2+1=0, which is SHORTER!
Dijkstra misses this because it already marked C as visited.
```

## When to Use Dijkstra

| Situation | Algorithm |
|-----------|-----------|
| Unweighted graph, shortest path | BFS |
| Weighted graph, no negative weights | Dijkstra |
| Negative weights possible | Bellman-Ford |
| All pairs shortest paths | Floyd-Warshall |

---
## Mini-Quiz

In [ ]:
# Q1: What data structure does Dijkstra use internally?
# Answer:

# Q2: What is the time complexity of Dijkstra?
# Answer:

# Q3: Can Dijkstra find shortest paths in an unweighted graph?
# Answer: Yes/No and why:

---
## Reflection (Required - Complete before submitting)

In the cell below, write 3-5 bullet points:
1. **What I learned today:**
2. **What was hardest:**
3. **What I still don't understand:**
4. **What I will review before next week:**
5. **If I used AI tools, what for:**

In [ ]:
# YOUR REFLECTION (write as comments or as a multi-line string)
reflection = """
- What I learned:
- What was hardest:
- What I still don't understand:
- What I will review:
- AI tools used (if any):
"""
print(reflection)